# Test Planner 
## Phase 1 - bmr_tdee, diet_logic, workout_logic

Interactive sanity checks for the core planner modules before wiring into Flask/MySQL.

Place this notebook at the project root (same level as the `modules/` folder).

In [1]:
import sys
sys.path.append('.')

from modules.bmr_tdee import calculate_bmr, calculate_tdee, calculate_bmi
from modules.diet_logic import get_diet_plan
from modules.workout_logic import get_workout_split

## 1. BMR / TDEE / BMI - normal cases

Check these against a trusted online TDEE calculator before trusting them.

In [2]:
test_users = [
    {"weight_kg": 70, "height_cm": 170, "age": 25, "gender": "male", "activity_level": "moderately_active"},
    {"weight_kg": 58, "height_cm": 160, "age": 22, "gender": "female", "activity_level": "lightly_active"},
    {"weight_kg": 85, "height_cm": 180, "age": 30, "gender": "male", "activity_level": "very_active"},
]

for user in test_users:
    bmr = calculate_bmr(user["weight_kg"], user["height_cm"], user["age"], user["gender"])
    tdee = calculate_tdee(bmr, user["activity_level"])
    bmi = calculate_bmi(user["weight_kg"], user["height_cm"])
    print(f"{user}")
    print(f"  BMR: {bmr:.1f} kcal | TDEE: {tdee:.1f} kcal | BMI: {bmi:.1f}\n")

{'weight_kg': 70, 'height_cm': 170, 'age': 25, 'gender': 'male', 'activity_level': 'moderately_active'}
  BMR: 1642.5 kcal | TDEE: 2545.9 kcal | BMI: 24.2

{'weight_kg': 58, 'height_cm': 160, 'age': 22, 'gender': 'female', 'activity_level': 'lightly_active'}
  BMR: 1309.0 kcal | TDEE: 1799.9 kcal | BMI: 22.7

{'weight_kg': 85, 'height_cm': 180, 'age': 30, 'gender': 'male', 'activity_level': 'very_active'}
  BMR: 1830.0 kcal | TDEE: 3156.8 kcal | BMI: 26.2



## 2. BMR / TDEE - edge cases (should raise ValueError)

Confirms your validation actually fires instead of silently defaulting.

In [3]:
# Casing/whitespace should still work (normalized internally)
print(calculate_bmr(70, 170, 25, "Male "))

# Genuinely invalid input should raise
try:
    calculate_bmr(70, 170, 25, "banana")
    print("FAILED: should have raised ValueError")
except ValueError as e:
    print(f"OK - correctly raised: {e}")

try:
    calculate_tdee(1600, "lightly active")  # space instead of underscore
    print("OK - normalized space to underscore")
except ValueError as e:
    print(f"FAILED unexpectedly: {e}")

try:
    calculate_tdee(1600, "super_active")  # not a real key
    print("FAILED: should have raised ValueError")
except ValueError as e:
    print(f"OK - correctly raised: {e}")

1642.5
OK - correctly raised: Invalid gender: 'banana'. Expected 'male' or 'female'.
OK - normalized space to underscore
OK - correctly raised: Invalid activity level: 'super_active'. Expected one of ['sedentary', 'lightly_active', 'moderately_active', 'very_active', 'extra_active'].


## 3. Diet plan - all three goals

Manually check: does protein_g×4 + carbs_g×4 + fat_g×9 land close to calorie_target?

In [4]:
sample_tdee = 2200

for goal in ["muscle_gain", "fat_loss", "maintenance"]:
    plan = get_diet_plan(sample_tdee, goal)
    check = plan["protein_g"] * 4 + plan["carbs_g"] * 4 + plan["fat_g"] * 9
    print(f"Goal: {goal}")
    print(f"  {plan}")
    print(f"  Reconstructed calories: {check:.0f} (target was {plan['calorie_target']})\n")

Goal: muscle_gain
  {'calorie_target': 2500, 'protein_g': 187.5, 'carbs_g': 281.2, 'fat_g': 69.4}
  Reconstructed calories: 2499 (target was 2500)

Goal: fat_loss
  {'calorie_target': 1700, 'protein_g': 170.0, 'carbs_g': 127.5, 'fat_g': 56.7}
  Reconstructed calories: 1700 (target was 1700)

Goal: maintenance
  {'calorie_target': 2200, 'protein_g': 165.0, 'carbs_g': 220.0, 'fat_g': 73.3}
  Reconstructed calories: 2200 (target was 2200)



## 4. Workout split - check logic across goal x experience combos

In [5]:
combos = [
    ("muscle_gain", "advanced"),
    ("fat_loss", "beginner"),
    ("maintenance", "intermediate"),
]

for goal, level in combos:
    print(f"=== Goal: {goal} | Experience: {level} ===")
    split = get_workout_split(goal, level)
    for day, details in split.items():
        if details["type"] == "rest":
            print(f"  {day}: Rest")
        else:
            print(f"  {day}: {details['type']} - {details['sets']}x{details['reps']}, rest {details['rest_seconds']}s")
            print(f"      Exercises: {', '.join(details['exercises'])}")
    print()

=== Goal: muscle_gain | Experience: advanced ===
  Monday: push - 4x8-12, rest 90s
      Exercises: Bench Press, Overhead Press, Incline Dumbbell Press, Tricep Pushdown
  Tuesday: pull - 4x8-12, rest 90s
      Exercises: Deadlift, Bent-over Row, Lat Pulldown, Face Pull, Bicep Curl
  Wednesday: legs - 4x8-12, rest 90s
      Exercises: Squat, Romanian Deadlift, Leg Press, Calf Raise
  Thursday: Rest
  Friday: push - 4x8-12, rest 90s
      Exercises: Bench Press, Overhead Press, Incline Dumbbell Press, Tricep Pushdown
  Saturday: pull - 4x8-12, rest 90s
      Exercises: Deadlift, Bent-over Row, Lat Pulldown, Face Pull, Bicep Curl
  Sunday: legs - 4x8-12, rest 90s
      Exercises: Squat, Romanian Deadlift, Leg Press, Calf Raise

=== Goal: fat_loss | Experience: beginner ===
  Monday: full_body - 3x12-15, rest 45s
      Exercises: Squat, Bench Press, Bent-over Row, Overhead Press, Plank
  Tuesday: Rest
  Wednesday: full_body - 3x12-15, rest 45s
      Exercises: Squat, Bench Press, Bent-over

## 5. Full end-to-end - one simulated user through all three modules

In [6]:
user = {
    "weight_kg": 65,
    "height_cm": 165,
    "age": 24,
    "gender": "female",
    "activity_level": "moderately_active",
    "goal": "fat_loss",
    "experience_level": "beginner"
}

bmr = calculate_bmr(user["weight_kg"], user["height_cm"], user["age"], user["gender"])
tdee = calculate_tdee(bmr, user["activity_level"])
bmi = calculate_bmi(user["weight_kg"], user["height_cm"])
diet_plan = get_diet_plan(tdee, user["goal"])
workout_split = get_workout_split(user["goal"], user["experience_level"])

print(f"User: {user}\n")
print(f"BMR: {bmr:.1f} | TDEE: {tdee:.1f} | BMI: {bmi:.1f}\n")
print(f"Diet plan: {diet_plan}\n")
print("Workout split:")
for day, details in workout_split.items():
    print(f"  {day}: {details['type']}")

User: {'weight_kg': 65, 'height_cm': 165, 'age': 24, 'gender': 'female', 'activity_level': 'moderately_active', 'goal': 'fat_loss', 'experience_level': 'beginner'}

BMR: 1400.2 | TDEE: 2170.4 | BMI: 23.9

Diet plan: {'calorie_target': 1670, 'protein_g': 167.0, 'carbs_g': 125.3, 'fat_g': 55.7}

Workout split:
  Monday: full_body
  Tuesday: rest
  Wednesday: full_body
  Thursday: rest
  Friday: full_body
  Saturday: rest
  Sunday: rest


## Notes / observations

_(Use this cell to jot down anything that looks off, or questions to revisit — good habit for a portfolio notebook.)_